# Actividad 3 – Cadenas y Lenguajes con FastAPI

**Estudiante:** Angel Rugerio Jiménez
**No. de cuenta:** 201720
**Actividad:** Actividad 3 – Operaciones de cadenas y lenguajes a través de un servicio web (FastAPI)

## Objetivo

Implementar las operaciones de cadenas y lenguajes vistas en clase como endpoints de un servicio
web con FastAPI, y usarlos para resolver los ejercicios propuestos sobre el alfabeto y los
lenguajes dados.

## Configuración

En lugar de levantar `uvicorn` como proceso aparte, este notebook usa
`fastapi.testclient.TestClient`, que manda las peticiones a través de la misma pila ASGI que usaría
un servidor real (routing, validación de Pydantic, serialización de la respuesta). Es la forma
estándar de probar una app de FastAPI y garantiza que los resultados que se muestran abajo son
exactamente los que producen los endpoints definidos en `main.py`.

In [8]:
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

def post(path, body):
    r = client.post(path, json=body)
    assert r.status_code == 200, (path, body, r.status_code, r.text)
    return r.json()

def get(path):
    r = client.get(path)
    assert r.status_code == 200, (path, r.status_code, r.text)
    return r.json()

def fmt_set(s):
    elems = sorted(s, key=lambda x: (len(x), x))
    elems = ["λ" if e == "" else e for e in elems]
    return "{" + ", ".join(elems) + "}"

def fmt_list(lst):
    elems = ["λ" if e == "" else e for e in lst]
    return "[" + ", ".join(elems) + "]"

print("Servicio listo:", get("/"))

Servicio listo: {'Hello': 'World'}


## Parte 1 – Operaciones con cadenas

Endpoints: `GET /Cadenas/concatenar/{x}/{y}`, `GET /Cadenas/unir/{x}/{y}`, `GET /Cadenas/potencia/{x}/{n}`.

Ejemplo con `x = "hola"`, `y = "mundo"`, `n = 3`.

In [2]:
concat = get("/Cadenas/concatenar/hola/mundo")
union_cadenas = get("/Cadenas/unir/hola/mundo")
potencia = get("/Cadenas/potencia/ab/3")

print("concatenar('hola', 'mundo') ->", concat["resultado"])
print("unir('hola', 'mundo')       ->", union_cadenas["resultado"])
print("potencia('ab', 3)           ->", potencia["resultado"])

concatenar('hola', 'mundo') -> holamundo
unir('hola', 'mundo')       -> ['hola', 'mundo']
potencia('ab', 3)           -> ababab


## Parte 2 – Ejercicios sobre lenguajes

Dado el alfabeto $\Sigma = \{a, b\}$ y los lenguajes:

- $L = \{\lambda, a, b\}$ (la cadena vacía $\lambda$ se representa como `""`)
- $M = \{b, aa\}$

se calculan las operaciones pedidas usando los endpoints `/lenguajes/*`.

In [3]:
L = ["", "a", "b"]
M = ["b", "aa"]

print("L =", fmt_set(L))
print("M =", fmt_set(M))

L = {λ, a, b}
M = {b, aa}


### Operaciones de conjuntos: L∪M, L∩M, L−M, M−L

In [4]:
r_union = post("/lenguajes/union", {"L": L, "M": M})
r_inter = post("/lenguajes/interseccion", {"L": L, "M": M})
r_dif_LM = post("/lenguajes/diferencia", {"L": L, "M": M})
r_dif_ML = post("/lenguajes/diferencia", {"L": M, "M": L})  # M-L: se invierten los argumentos

print("L U M =", fmt_set(r_union["resultado"]))
print("L ^ M =", fmt_set(r_inter["resultado"]))
print("L - M =", fmt_set(r_dif_LM["resultado"]))
print("M - L =", fmt_set(r_dif_ML["resultado"]))

L U M = {λ, a, b, aa}
L ^ M = {b}
L - M = {λ, a}
M - L = {aa}


### Operaciones sobre cadenas (concatenación de lenguajes): L.M, M.L, M²

`/lenguajes/concatenar` calcula $\{xy : x \in L, y \in M\}$, así que para $M^2$ se manda $M$
como ambos parámetros.

In [5]:
r_LM = post("/lenguajes/concatenar", {"L": L, "M": M})
r_ML = post("/lenguajes/concatenar", {"L": M, "M": L})
r_M2 = post("/lenguajes/concatenar", {"L": M, "M": M})
r_L2 = post("/lenguajes/concatenar", {"L": L, "M": L})  # se reutiliza más abajo

print("L.M =", fmt_set(r_LM["resultado"]))
print("M.L =", fmt_set(r_ML["resultado"]))
print("M^2 =", fmt_set(r_M2["resultado"]))
print("L^2 =", fmt_set(r_L2["resultado"]))

L.M = {b, aa, ab, bb, aaa, baa}
M.L = {b, aa, ba, bb, aaa, aab}
M^2 = {bb, aab, baa, aaaa}
L^2 = {λ, a, b, aa, ab, ba, bb}


### Clausura de Kleene y combinación: L*, M*, (LM) ∪ (M*∩L²)

`/lenguajes/kleene` recibe `k` como número de iteraciones (no como longitud máxima), así que se
pide con un `k` generoso (8) y del resultado —que ya es la clausura hasta esa iteración— se
extraen los primeros 8 elementos en **orden shortlex** (primero por longitud, luego alfabético),
que es el orden estándar para enumerar un lenguaje infinito.

Para la combinación final se encadenan los resultados de endpoints ya calculados:
$L \cdot M$ y $L^2$ (arriba) con $M^*$.

In [6]:
def shortlex_first8(elementos):
    return sorted(elementos, key=lambda s: (len(s), s))[:8]

r_Lstar = post("/lenguajes/kleene", {"L": L, "k": 8})
r_Mstar = post("/lenguajes/kleene", {"L": M, "k": 8})

L_estrella_8 = shortlex_first8(r_Lstar["resultado"])
M_estrella_8 = shortlex_first8(r_Mstar["resultado"])

print("L* (primeros 8) =", fmt_list(L_estrella_8))
print("M* (primeros 8) =", fmt_list(M_estrella_8))

L* (primeros 8) = [λ, a, b, aa, ab, ba, bb, aaa]
M* (primeros 8) = [λ, b, aa, bb, aab, baa, bbb, aaaa]


In [7]:
# (L.M) U (M* ∩ L^2)
r_inter_final = post("/lenguajes/interseccion", {"L": r_Mstar["resultado"], "M": r_L2["resultado"]})
r_final = post("/lenguajes/union", {"L": r_LM["resultado"], "M": r_inter_final["resultado"]})

print("M* n L^2          =", fmt_set(r_inter_final["resultado"]))
print("(LM) U (M* n L^2) =", fmt_set(r_final["resultado"]))

M* n L^2          = {λ, b, aa, bb}
(LM) U (M* n L^2) = {λ, b, aa, ab, bb, aaa, baa}


## Conclusión

Todos los resultados anteriores fueron producidos exclusivamente por los endpoints de
`main.py` (operaciones de conjuntos, concatenación de lenguajes y clausura de Kleene), encadenando
llamadas cuando el ejercicio lo requería (por ejemplo, $L^2$ y $M^*$ se calcularon una sola vez y
se reutilizaron para la combinación final). Los resultados formateados se replican en
`README.md`.